# Banking Customer Default Risk Classification

## Academic Project

### Business objective
Develop a classification model to identify customers with higher observed risk of loan default using available customer, account and demographic attributes.

### Business context
Banks need to supplement traditional credit assessment with additional customer information. This project explores whether demographic and banking attributes can improve identification of customers associated with default outcomes.

> **Important:** This is an academic modelling exercise. The results should not be used as a production lending decision system without validation, fairness assessment, regulatory review, explainability controls and out-of-sample monitoring.


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

data = pd.read_csv("../data/Banking.csv")
data.head()

## 1. Data Preparation

The dataset is inspected for duplicates, missing values, low-information columns and high-cardinality numerical variables. Categorical variables are encoded for model development.

In [ ]:
from src.data_preprocessing import (
    feature_engineering,
    label_encoding,
    missing_data_threshold,
    missing_data_treatment,
    unique_value_treatment,
    resampling,
)

print("Initial shape:", data.shape)
data = data.drop_duplicates(keep="first")
print("After duplicate removal:", data.shape)

data = missing_data_threshold(data, threshold=75)
data = missing_data_treatment(data)
data = unique_value_treatment(data)
data = feature_engineering(data)
data = label_encoding(data)

## 2. Target Imbalance

The default class is much smaller than the non-default class. The academic exercise therefore uses minority-class resampling before model development.

In [ ]:
print(data["DefaulterYN"].value_counts())
data = resampling(data, "DefaulterYN", minority_class_value=1, multiply=15)
print(data["DefaulterYN"].value_counts())

In [ ]:
x = data.drop(columns="DefaulterYN")
y = data["DefaulterYN"]

from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    x, y, train_size=0.7, random_state=42, stratify=y
)

## 3. Model Development & Comparison

Multiple classification algorithms are compared using the same train/test split. Recall is especially relevant for this academic risk-screening use case because missing a true defaulter and incorrectly flagging a non-defaulter have different business consequences.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

models = [
    LogisticRegression(max_iter=1000),
    KNeighborsClassifier(),
    DecisionTreeClassifier(random_state=42),
    RandomForestClassifier(random_state=42),
    AdaBoostClassifier(random_state=42),
    GradientBoostingClassifier(random_state=42),
]

results = []
for model in models:
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    results.append({
        "model": model.__class__.__name__,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    })

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
results_df

## 4. Business Interpretation

- **Recall:** proportion of actual defaulters identified by the model.
- **Precision:** proportion of predicted defaulters who are actually defaulters.
- **F1:** balance between precision and recall.
- **Confusion matrix:** shows the operational trade-off between false positives and false negatives.

A production implementation would additionally require probability calibration, threshold optimization, explainability, fairness testing and temporal/out-of-sample validation.

In [ ]:
# Example confusion matrix for the best-performing model after business-driven model selection.
# Add a selected model here after validating the comparison above.


## Portfolio Takeaway

The project demonstrates a business-oriented machine learning workflow: translate a risk question into a classification problem, address class imbalance, compare models using decision-relevant metrics, and communicate the limitations before considering any real-world use.